In [123]:
import pandas as pd
from edgar import *
import yfinance as yf
import os
import numpy as np
from dotenv import load_dotenv
load_dotenv()
set_identity(os.getenv("EMAIL"))
print("All imported")

All imported


In [144]:
def get_company_dcf_data(ticker, verbose= False):
    
    company = Company(ticker)
    financials = company.get_financials()
    df_inc  = company.income_statement().to_dataframe()
    df_cash = company.cashflow_statement().to_dataframe()
    df_bal=company.balance_sheet().to_dataframe()
    yfin_ticker = yf.Ticker(ticker)
    info   = yfin_ticker.info

    # ── Find year column (handle FY and non-FY formats) ──────────────
    year_cols = [c for c in df_inc.columns if "FY" in c]
    if not year_cols:
        # fallback: take first numeric-looking column
        year_cols = [c for c in df_inc.columns if any(ch.isdigit() for ch in str(c))]
    if not year_cols:
        print(f" {ticker}: no year columns found. Columns: {df_inc.columns.tolist()}")
        return None
    year = year_cols[0]
    year_prev= year_cols[1]
    

    # ── Operating Income ─────────────────────────────────────────────
    a = df_inc.loc[df_inc.index.str.contains("OperatingIncomeLoss", na=False), year]
    operating_income = int(a.iloc[0]) if not a.empty else 0

    # ── D&A ──────────────────────────────────────────────────────────
    dep_am = 0
    for tag in ["DepreciationDepletionAndAmortization", "DepreciationAndAmortization", "Depreciation"]:
        a = df_cash.loc[df_cash.index.str.contains(tag, na=False), year]
        if not a.empty and pd.notna(a.iloc[0]):
            dep_am = a.iloc[0]
            break

    # ── EBITDA ───────────────────────────────────────────────────────
    ebitda = operating_income + dep_am

    
    # ── CAPEX ───────────────────────────────────────────────────────
    CAPEX = abs(yfin_ticker.cashflow.loc["Capital Expenditure"].iloc[0] )
    for tag in ["PaymentsToAcquirePropertyPlantAndEquipment","CapitalExpenditures"]:
        a = df_cash.loc[df_cash.index.str.contains(tag, na=False), year]
        if not a.empty and pd.notna(a.iloc[0]):
            CAPEX = a.iloc[0]
            break
    
    # ── NWC ───────────────────────────────────────────────────────
    assets_m = df_bal.loc[df_bal.index == "AssetsCurrent"]
    liab_m   = df_bal.loc[df_bal.index == "LiabilitiesCurrent"]
    NWC = (assets_m[year].iloc[0] - liab_m[year].iloc[0]) if (not assets_m.empty and not liab_m.empty) else None
    NWC_1 = (assets_m[year_prev].iloc[0] - liab_m[year_prev].iloc[0]) if (not assets_m.empty and not liab_m.empty) else None
    NWC_change= (NWC - NWC_1)
    
    # ── TAX RATE ───────────────────────────────────────────────────────
    tax_m    = df_inc.loc[df_inc.index == "IncomeTaxExpenseBenefit"]
    pretax_m = df_inc.loc[df_inc.index.str.contains("IncomeLossFromContinuingOperationsBeforeIncomeTaxes", na=False)]
    tax_rate = (tax_m[year].iloc[0] / pretax_m[year].iloc[0]) if (not tax_m.empty and not pretax_m.empty and pretax_m[year].iloc[0] != 0) else None
    tax_rate = 0.21 if (tax_rate < 0 or tax_rate > 0.40) else tax_rate
    
    
    # ──WACC EXTRA DATA ─────────────────────────────────────────────────────── 
    try:
        beta = round(calculate_beta(ticker),2)
    except:
        beta= info["beta"] 
        
    market_cap = info["marketCap"]
    total_debt = yfin_ticker.balance_sheet.loc["Total Debt"].iloc[0]
    
    try:
        interest_expense    = yfin_ticker.financials.loc["Interest Expense"].iloc[0]
        interest_expense = 0 if np.isnan(interest_expense) else abs(interest_expense)
    except:
        interest_expense = 0
        print(f"interest expense missing — so 0 used for {ticker}")
    
    
    # ──TRESURY BOND PRICE ───────────────────────────────────────────────────────  
    tnx = yf.Ticker("^TNX")
    data = tnx.history(period="1d")
    bond_10yr= data["Close"].iloc[-1]/100
    
    # ──CASH ─────────────────────────────────────────────────────── 
    cash = yfin_ticker.balance_sheet.loc["Cash And Cash Equivalents"].iloc[0]
    shares= info["sharesOutstanding"]
    share_price= info["previousClose"]
    
    confidence = "high" if (operating_income != 0 and dep_am != 0 and "2024"  not in str(year)) else "low"
        
    if verbose:   
        
        print(f"Using          {ticker}, {year} financial year")
        print(f"Confidence:    {confidence}, \n")
        
        print(f"Market cap:    {market_cap/1e6:.1f}M")
        print(f"Shares:        {shares}")
        print(f"Share price:   {share_price}")
        print(f"Cash:          {cash/1e6:.1f}M")
        print(f"InterestExp:   {interest_expense/1e6:.1f}M,\n")
        print(f"Total debt:    {total_debt/1e6:.1f}M,\n")
        
        
        print(f"EBITDA:        {ebitda/1e6:.1f}M")
        print(f"Ebit:          {operating_income/1e6:.1f}M")
        print(f"D&A:           {dep_am/1e6:.1f}M ")
        print(f"CAPEX:         {CAPEX/1e6:.1f}M")
        print(f"NWC change:    {NWC_change/1e6:.1f}M")
        print(f"tax_rate:      {tax_rate:.1%}")
        print("-"*50)
        print("")


    return {
        "operating_income": operating_income,
        "dep_am":           dep_am,
        "ebitda":           ebitda,
        "CAPEX":            CAPEX,
        "NWC":              NWC,
        "NWC_change":       NWC_change,
        "tax_rate":         tax_rate,
        "cash":             cash,
        "market_cap":       market_cap,
        "shares":           shares,
        "share_price":      share_price,
        "beta":             beta,
        "total_debt":       total_debt,
        "interest_expense": interest_expense,
        "bond_10yr":        bond_10yr,
        "year":             year,
        "confidence":       confidence
    }

In [63]:
def forecast_unlevered_FCF(c_data, year= 5, g= .05):
    
    fcf_list=[]
    
    FCF=  c_data["operating_income"] * (1- c_data["tax_rate"]) + c_data["dep_am"] + c_data["NWC_change"] - c_data["CAPEX"]
    print(f"FCF:        {FCF/1e6:.1f}M")
    
    for i in range (1, year+1):   
        yr_ebit= c_data["operating_income"] * (1+g)**i
        yr_fcf= yr_ebit * (1- c_data["tax_rate"]) + c_data["dep_am"] + c_data["NWC_change"] - c_data["CAPEX"]
        fcf_list.append(yr_fcf)
    
    print(fcf_list)
    
    return fcf_list

In [ ]:
tnx = yf.Ticker("^TNX")
data = tnx.history(period="1y")
data["Close"].iloc[-1]

4.440000057220459

In [142]:
a= get_company_dcf_data("AAPL")
for i in a:
    print(i)

Using       yfinance.Ticker object <AAPL>, FY 2025 financial year
EBITDA:     144748.0M
Ebit:       133050.0M
D&A:        11698.0M 
CAPEX:      12715.0M
NWC:        -17674.0M
tax_rate:   15.6%
operating_income
dep_am
ebitda
CAPEX
NWC
NWC_change
tax_rate
cash
market_cap
shares
beta
total_debt
interest_expense
bond_10yr
year
confidence


In [ ]:
def get_wacc(c_data, equity_risk_premium=.05):
    
    a["bond_10yr"]
    
    cost_of_equity = a["bond_10yr"]+ a["beta"]* equity_risk_premium

    cost_debt= a["interest_expense"]/a["total_debt"]
    cost_debt_aftertax= cost_debt * (1-a["tax_rate"])

    enterprise_value= a["market_cap"] + a["total_debt"]

    WACC = (a["market_cap"] / enterprise_value) * cost_of_equity + (a["total_debt"] / enterprise_value) * cost_debt_aftertax

    return WACC
   
    print (cost_of_equity)
    print(cost_debt)
    print(cost_debt_aftertax)
    print(enterprise_value)
    print(WACC)


0.1002000005722046
0.0
0.0
3755501050432.0
0.0975677468932142


In [64]:
c_data= get_company_dcf_data("AAPL")
FCF= forecast_unlevered_FCF(c_data)


Using       AAPL, FY 2025 financial year
EBITDA:     144748.0M
Ebit:       133050.0M
D&A:        11698.0M 
CAPEX:      12715.0M
NWC:        -17674.0M
tax_rate:   15.6%
FCF:        116994.9M
[122608936487.1279, 128503683311.48431, 134693167477.05853, 141192125850.91147, 148016032143.45706]


In [69]:
company = Company("AAPL")
financials = company.get_financials()
df_inc  = company.income_statement().to_dataframe()
df_cash = company.cashflow_statement().to_dataframe()
df_bal=company.balance_sheet().to_dataframe()

In [ ]:
a= yf.Ticker("SONO")
b= a.financials



,2025-09-30,2024-09-30,2023-09-30,2022-09-30
Tax Effect Of Unusual Items,-3.678150e+06,-1.283100e+06,0.000000e+00,0.000000e+00
Tax Rate For Calcs,2.100000e-01,2.100000e-01,2.100000e-01,1.959800e-02
Normalized EBITDA,2.980400e+07,3.177800e+07,5.409600e+07,1.077860e+08
Total Unusual Items,-1.751500e+07,-6.110000e+06,-9.093000e+06,0.000000e+00
Total Unusual Items Excluding Goodwill,-1.751500e+07,-6.110000e+06,-9.093000e+06,0.000000e+00
Net Income From Continuing Operation Net Minority Interest,-6.114400e+07,-3.814600e+07,-1.027400e+07,6.738300e+07
Reconciled Depreciation,6.232100e+07,5.237800e+07,4.896900e+07,3.850400e+07
Reconciled Cost Of Revenue,8.127460e+08,8.286830e+08,9.387650e+08,9.559690e+08
EBITDA,1.228900e+07,2.566800e+07,5.409600e+07,1.077860e+08
EBIT,-5.003200e+07,-2.671000e+07,5.127000e+06,6.928200e+07


In [127]:
us_stocks = [
    "AAPL",   # Apple
    "MSFT",   # Microsoft
    "NVDA",   # Nvidia
    "AMZN",   # Amazon
    "GOOGL",  # Alphabet A
    "META",   # Meta Platforms
    "TSLA",   # Tesla
    
    "JPM",    # JPMorgan
    "V",      # Visa
    "MA",     # Mastercard
    "XOM",    # Exxon Mobil
    "WMT",    # Walmart
    "LLY",    # Eli Lilly
    "JNJ",    # Johnson & Johnson
    "PG",     # Procter & Gamble
    "HD",     # Home Depot
    "BAC",    # Bank of America
    "NFLX",   # Netflix
    "ORCL"    # Oracle
]
for i in us_stocks:  
    ticker = yf.Ticker(i)
    info   = ticker.info
    market_cap = info["marketCap"]
    total_debt = ticker.balance_sheet.loc["Total Debt"].iloc[0]    
    interest_expense    = ticker.financials.loc["Interest Expense"].iloc[0]
    
    interest_expense= 0 if np.isnan(interest_expense) else interest_expense
    total_debt= 0 if np.isnan( total_debt) else  total_debt
    print(f"{interest_expense}.     {market_cap}.    {total_debt}")

0.     3656844050432.    98657000000.0
2385000000.0.     2651649474560.    60588000000.0
259000000.0.     4071573684224.    11040000000.0
2274000000.0.     2139899035648.    152987000000.0
736000000.0.     3318691135488.    59291000000.0
1165000000.0.     1329837768704.    83897000000.0
338000000.0.     1357742342144.    14719000000.0
97898000000.0.     762828619776.    499982000000.0
589000000.0.     569774243840.    25171000000.0
722000000.0.     432154017792.    19000000000.0
603000000.0.     712474886144.    43537000000.0
2799000000.0.     979728531456.    67095000000.0
0.     786041274368.    42503000000.0
971000000.0.     579460202496.    47933000000.0
907000000.0.     333475250176.    35463000000.0
0.     320367067136.    62290000000.0
78470000000.0.     337088774144.    365904000000.0
776510000.0.     396319752192.    14462836000.0
3578000000.0.     401668603904.    104104000000.0


In [137]:
print(yf.Ticker("AAPL").financials.index.tolist())

['Tax Effect Of Unusual Items', 'Tax Rate For Calcs', 'Normalized EBITDA', 'Net Income From Continuing Operation Net Minority Interest', 'Reconciled Depreciation', 'Reconciled Cost Of Revenue', 'EBITDA', 'EBIT', 'Net Interest Income', 'Interest Expense', 'Interest Income', 'Normalized Income', 'Net Income From Continuing And Discontinued Operation', 'Total Expenses', 'Total Operating Income As Reported', 'Diluted Average Shares', 'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Diluted NI Availto Com Stockholders', 'Net Income Common Stockholders', 'Net Income', 'Net Income Including Noncontrolling Interests', 'Net Income Continuous Operations', 'Tax Provision', 'Pretax Income', 'Other Income Expense', 'Other Non Operating Income Expenses', 'Net Non Operating Interest Income Expense', 'Interest Expense Non Operating', 'Interest Income Non Operating', 'Operating Income', 'Operating Expense', 'Research And Development', 'Selling General And Administration', 'Gross Profit', 'Cost Of R

In [ ]:
def terminal_value(fcf_list, WACC, g_longterm= .025):
    
    return (fcf_list[-1]*(1 + g_longterm) /(WACC - g_longterm))

In [ ]:
def discount_FCF_WACC(fcf_list, WACC, g=.8, g_longterm= .025 ):
    
    fcf_projected= [fcf / (1 + WACC)** i for  i , fcf in enumerate (fcf_list, 1)] 
    TV      = terminal_value(fcf_projected[-1], WACC)
    pv_TV   = TV / (1 + WACC) ** 5
    enterprise_value      = sum(fcf_projected) + pv_TV
    
    return enterprise_value

In [141]:
def final_details(c_data, enterprise_value):
    
    equity_value= enterprise_value - c_data["total_debt"]+ c_data["cash"]
    intrinsic_price= equity_value/ c_data["shares"]
    
    return intrinsic_price

In [143]:
yfin_ticker = yf.Ticker("AAPL")
info   = yfin_ticker.info

for i in info: print(i)

address1
city
state
zip
country
phone
website
industry
industryKey
industryDisp
sector
sectorKey
sectorDisp
longBusinessSummary
fullTimeEmployees
companyOfficers
auditRisk
boardRisk
compensationRisk
shareHolderRightsRisk
overallRisk
governanceEpochDate
compensationAsOfEpochDate
irWebsite
executiveTeam
maxAge
priceHint
previousClose
open
dayLow
dayHigh
regularMarketPreviousClose
regularMarketOpen
regularMarketDayLow
regularMarketDayHigh
dividendRate
dividendYield
exDividendDate
payoutRatio
fiveYearAvgDividendYield
beta
trailingPE
forwardPE
volume
regularMarketVolume
averageVolume
averageVolume10days
averageDailyVolume10Day
bid
ask
bidSize
askSize
marketCap
nonDilutedMarketCap
fiftyTwoWeekLow
fiftyTwoWeekHigh
allTimeHigh
allTimeLow
priceToSalesTrailing12Months
fiftyDayAverage
twoHundredDayAverage
trailingAnnualDividendRate
trailingAnnualDividendYield
currency
tradeable
enterpriseValue
profitMargins
floatShares
sharesOutstanding
sharesShort
sharesShortPriorMonth
sharesShortPreviousMonthDa

In [151]:
def save_proyects_data():
    
    companies=[]
    tickers = ["META", "AAPL", "MSFT"]
    for t in tickers:
        data = get_company_dcf_data(t, verbose=False) 
        data["ticker"] = t  # add ticker column# your function
        companies.append(data)
        
    df = pd.DataFrame(companies)
    cols = ["ticker"] + [c for c in df.columns if c != "ticker"]
    df = df[cols]
    print(df)
    # Save to CSV
    df.to_csv("DCF_data.csv", index=False)

In [152]:
save_proyects_data()

  ticker  operating_income        dep_am        ebitda         CAPEX  \
0   META       83276000000  1.861600e+10  1.018920e+11  6.969100e+10   
1   AAPL      133050000000  1.169800e+10  1.447480e+11  1.271500e+10   
2   MSFT      128528000000  2.200000e+10  1.505280e+11  6.455100e+10   

            NWC    NWC_change  tax_rate          cash     market_cap  \
0  6.688600e+10  4.370000e+08  0.296444  3.587300e+10  1447234633728   
1 -1.767400e+10  5.731000e+09  0.156100  3.593400e+10  3730186436608   
2  4.991300e+10  1.546500e+10  0.176296  3.024200e+10  2751243485184   

        shares  share_price   beta    total_debt  interest_expense  bond_10yr  \
0   2187177748       536.38  1.279  8.389700e+10      1.165000e+09    0.04311   
1  14681140000       246.63  1.116  9.865700e+10      0.000000e+00    0.04311   
2   7425629076       358.96  1.108  6.058800e+10      2.385000e+09    0.04311   

      year confidence  
0  FY 2025       high  
1  FY 2025       high  
2  FY 2025       high  
